In [1]:
# Ignore annoying warning from astropy fits reader about wrong units in the FITS files
%env PYTHONWARNINGS=ignore

env: PYTHONWARNINGS=ignore


In [2]:
from itertools import chain
from pathlib import Path

import pyarrow as pa
from astropy.io import fits
from hats_import import CollectionArguments, pipeline_with_client
from hats_import.catalog.file_readers import InputReader, FitsReader
from dask.distributed import Client

/ocean/projects/phy210048p/malanche/hats-import-pipelines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path('/ocean/projects/phy210048p/shared/hats/')

DR = '10.1'
HATS_ROOT = ROOT / 'catalogs'
assert HATS_ROOT.exists()
CAT_NAME = f"legacysurvey_dr{DR}"
OUTPUT_DIR = HATS_ROOT / CAT_NAME

INPUT_ROOT = ROOT / 'raw' / 'legacysurvey'
MAIN_INPUT_DIR = INPUT_ROOT / DR
assert MAIN_INPUT_DIR.exists()

In [4]:
some_main_input_path = next(MAIN_INPUT_DIR.glob("*.fits"))
main_cat_columns = fits.open(some_main_input_path)[1].columns.names

In [5]:
class LegacySurveyReader(InputReader):
    def __init__(self, chunksize=100_000, supplementary_catalogs=None):
        self.chunksize = chunksize
        self.supplementary_catalogs = supplementary_catalogs
        self.fits_reader = FitsReader(chunksize=self.chunksize)
    
    def read(self, input_file, read_columns=None):
        if read_columns is not None or self.supplementary_catalogs is None:
            yield from self.fits_reader.read(input_file, read_columns=read_columns)
            return
        
        main_gen = self.fits_reader.read(input_file, read_columns=read_columns)
        suppl_gens = []
        for dir_path, file_suffix in self.supplementary_catalogs:
            suppl_catalog_fname = f'{input_file.stem}{file_suffix}.fits'
            suppl_catalog_path = dir_path / suppl_catalog_fname
            suppl_gens.append(
                self.fits_reader.read(suppl_catalog_path, read_columns=read_columns),
            )
        
        for main_table, *suppl_tables in zip(main_gen, *suppl_gens):
            for suppl_table_len in map(len, suppl_tables):
                assert suppl_table_len == len(main_table)
            # Remove suppl columns which are in main table
            for i, supplt_table in enumerate(suppl_tables):
                remove_columns = set(supplt_table.schema.names) & set(main_table.schema.names)
                suppl_tables[i] = supplt_table.drop(remove_columns)
            concat_table = pa.Table.from_arrays(
                list(main_table.columns) + list(chain.from_iterable(t.columns for t in suppl_tables)),
                names=list(main_table.schema.names) + list(chain.from_iterable(t.schema.names for t in suppl_tables)),
            )
            
            yield concat_table


In [ ]:
args = CollectionArguments(
    output_artifact_name=CAT_NAME,
    output_path=HATS_ROOT,
    # output_path='hats',
).catalog(
    input_file_list=sorted(MAIN_INPUT_DIR.glob("*.fits")),
    # input_file_list=[MAIN_INPUT_DIR / 'sweep-000m005-005p000.fits'],
    pixel_threshold=100_000,
    file_reader=LegacySurveyReader(
        chunksize=10_000,
        supplementary_catalogs=[
            # folder path, file suffix
            (INPUT_ROOT / f"{DR}-extra", '-ex'),
            (INPUT_ROOT / f"{DR}-lightcurves", '-lc'),
            (INPUT_ROOT / f"{DR}-photo-z", '-pz'),
        ],
    ),
    ra_column="RA",
    dec_column="DEC",
    sort_columns="OBJID",
    addl_hats_properties={
        'hats_cols_default': main_cat_columns,
    },
).add_margin(
    margin_threshold=10.0,
    is_default=True
).add_index(
    indexing_column="OBJID",
    include_healpix_29=True,
    include_order_pixel=True,
    drop_duplicates=False,
)

with Client(n_workers=8, threads_per_worker=1, memory_limit=f"{256//4}GB") as client:
    display(client)
    pipeline_with_client(args, client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 8,Total memory: 476.84 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33207,Workers: 8
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 476.84 GiB
Comm: tcp://127.0.0.1:40679,Total threads: 1
Dashboard: http://127.0.0.1:37857/status,Memory: 59.60 GiB
Nanny: tcp://127.0.0.1:36687,


/ocean/projects/phy210048p/malanche/hats-import-pipelines/venv/lib/python3.12/site-packages/distributed/client.py:3383: UserWarning: Sending large graph of size 191.20 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
import lsdb

cat = lsdb.open_catalog(HATS_ROOT / CAT_NAME, columns='all')
cat